# Ratio Plot after JSMR

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import vector

import HH4b.plotting as plotting
import HH4b.utils as utils
from HH4b.utils import ShapeVar

In [ ]:
def make_vector(events: pd.DataFrame, obj: str):
    """Create a ``vector`` object from the columns of the dataframe"""
    mstring = "PNetMass" if obj == "ak8FatJet" else "Mass"

    return vector.array(
        {
            "pt": events[f"{obj}Pt"],
            "phi": events[f"{obj}Phi"],
            "eta": events[f"{obj}Eta"],
            "M": events[f"{obj}{mstring}"],
            "tau21": events["ak8FatJetTau2OverTau1"],
        }
    )

## Load Dataset

In [ ]:
year = "2022EE"  #
dir_name = "24Apr22_v12_signal"
path_to_dir = f"/eos/uscms/store/user/haoyang/bbbb/ttSkimmer/{dir_name}"

In [ ]:
# Load your dataset
samples = {
    "tt": ["TTto2L2Nu", "TTto4Q", "TTtoLNu2Q"],
}

dirs = {path_to_dir: samples}

filters = None

# columns to load
# the parquet files are too big so we can only load a few columns at a time without consumming much memory
load_columns = [
    ("weight", 1),
    ("ak8FatJetTau2OverTau1", 2),
    ("ak8FatJetMsd", 2),
    ("ak8FatJetPNetMass", 2),
    ("ak8FatJetEta", 2),
    ("ak8FatJetPhi", 2),
    ("ak8FatJetPt", 2),
    ("bbFatJetTopMatch", 2),
    ("bbFatJetNumQMatchedTop1", 2),
    ("bbFatJetNumQMatchedTop2", 2),
    ("bbFatJetNumBMatchedTop1", 2),
    ("bbFatJetNumBMatchedTop2", 2),
    ("finalWeight", 0),
]
# reformat into ("column name", "idx") format for reading multiindex columns
columns = []
for key, num_columns in load_columns:
    for i in range(num_columns):
        columns.append(f"('{key}', '{i}')")


events_dict = {}
for input_dir, samples in dirs.items():
    events_dict = {
        **events_dict,
        # this function will load files (only the columns selected), apply filters and compute a weight per event
        **utils.load_samples(
            input_dir, samples, year, filters=filters, columns=columns, reorder_legacy_txbb=False
        ),
    }

samples_loaded = list(events_dict.keys())
keys_loaded = list(events_dict[samples_loaded[0]].keys())
print("Keys in events_dict")
for i in keys_loaded:
    print(i)

In [ ]:
# Load your dataset
samples = {
    "muon": [
        "Muon_Run2022E",
        "Muon_Run2022F",
        "Muon_Run2022G",
    ],
}

dirs = {path_to_dir: samples}

filters = None

# columns to load
# the parquet files are too big so we can only load a few columns at a time without consumming much memory
load_columns = [
    ("weight", 1),
    ("ak8FatJetTau2OverTau1", 2),
    ("ak8FatJetMsd", 2),
    ("ak8FatJetPNetMass", 2),
    ("ak8FatJetEta", 2),
    ("ak8FatJetPhi", 2),
    ("ak8FatJetPt", 2),
    ("finalWeight", 0),
]
# reformat into ("column name", "idx") format for reading multiindex columns
columns = []
for key, num_columns in load_columns:
    for i in range(num_columns):
        columns.append(f"('{key}', '{i}')")


events_dict_data = {}
for input_dir, samples in dirs.items():
    events_dict_data = {
        **events_dict,
        # this function will load files (only the columns selected), apply filters and compute a weight per event
        **utils.load_samples(
            input_dir, samples, year, filters=filters, columns=columns, reorder_legacy_txbb=False
        ),
    }

samples_loaded = list(events_dict.keys())
keys_loaded = list(events_dict[samples_loaded[0]].keys())
print("Keys in events_dict")
for i in keys_loaded:
    print(i)

## Event cuts

In [ ]:
def cut_on_W_mass(events, greater_than=60):
    # get W jets per events
    fatjets = make_vector(events, "ak8FatJet")
    sort_by_fj_pt = np.argsort(fatjets.pt, axis=1)[:, ::-1]
    fj_sorted = np.take_along_axis(fatjets, sort_by_fj_pt, axis=1)
    leading_fj = fj_sorted[:, 0]
    return events[greater_than < leading_fj.M]

In [ ]:
# Higgs candidate selection example
events_data = events_dict_data["muon"]
events_data = cut_on_W_mass(events_data, greater_than=60)

In [ ]:
events_mc = events_dict["tt"]
events_mc = cut_on_W_mass(events_mc, greater_than=60)

In [ ]:
# AK4OutsideJet pt cut
# jets_outside_raw = make_vector(events_raw, "ak4JetOutside")
# j3_raw = jets_outside_raw[:, 0]
# j4_raw = jets_outside_raw[:, 1]
# j3j4_pt_cut = (j3_raw.pt > 20) & (j4_raw.pt > 20)

In [ ]:
len(events_data)

Before W mass cut: 96443

After W mass cut: 

## Define different matching categories

In [ ]:
# derive fatjet attributes
# use != as XOR
has_2_daughter_qs = np.array(events_mc["bbFatJetNumQMatchedTop1"] == 2) != np.array(
    events_mc["bbFatJetNumQMatchedTop2"] == 2
)
has_1_b = np.array(events_mc["bbFatJetNumBMatchedTop1"] == 1) != np.array(
    events_mc["bbFatJetNumBMatchedTop2"] == 1
)

In [ ]:
top_matched = (has_2_daughter_qs) & (has_1_b)
W_matched = (has_2_daughter_qs) & (~has_1_b)
unmatched = ~has_2_daughter_qs

## Select Leading Fatjet by pT

In [ ]:
fatjets_mc = make_vector(events_mc, "ak8FatJet")
mc_sort_by_fj_pt = np.argsort(fatjets_mc.pt, axis=1)[:, ::-1]
fj_sorted_mc = np.take_along_axis(fatjets_mc, mc_sort_by_fj_pt, axis=1)
leading_fj_mc = fj_sorted_mc[:, 0]

In [ ]:
fatjets_data = make_vector(events_data, "ak8FatJet")
data_sort_by_fj_pt = np.argsort(fatjets_data.pt, axis=1)[:, ::-1]
fj_sorted_data = np.take_along_axis(fatjets_data, data_sort_by_fj_pt, axis=1)
leading_fj_data = fj_sorted_data[:, 0]

## Sort leading fatjets tau21 into each category

In [ ]:
top_matched_sorted = np.take_along_axis(top_matched, mc_sort_by_fj_pt, axis=1)
leading_fj_mc_is_top_matched = top_matched_sorted[:, 0]
leading_fj_mc_top = leading_fj_mc[leading_fj_mc_is_top_matched]

In [ ]:
events_mc.loc[leading_fj_mc_is_top_matched, "leading_fj_tau21"] = leading_fj_mc_top["tau21"]

In [ ]:
W_matched_sorted = np.take_along_axis(W_matched, mc_sort_by_fj_pt, axis=1)
leading_fj_mc_is_W_matched = W_matched_sorted[:, 0]
leading_fj_mc_W = leading_fj_mc[leading_fj_mc_is_W_matched]

In [ ]:
events_mc.loc[leading_fj_mc_is_W_matched, "leading_fj_tau21"] = leading_fj_mc_W["tau21"]

In [ ]:
unmatched_sorted = np.take_along_axis(unmatched, mc_sort_by_fj_pt, axis=1)
leading_fj_mc_is_unmatched = unmatched_sorted[:, 0]
leading_fj_mc_unmatched = leading_fj_mc[leading_fj_mc_is_unmatched]

In [ ]:
events_mc.loc[leading_fj_mc_is_unmatched, "leading_fj_tau21"] = leading_fj_mc_unmatched["tau21"]

In [ ]:
events_data.loc[:, "leading_fj_tau21"] = leading_fj_data["tau21"]
events_data.loc[:, "finalWeight"] = 1

## Define Events by their leading fj matching

In [ ]:
# parse the events df to a way that util can accept
events_dict = {}
events_dict["data"] = events_data
events_dict["top_matched"] = events_mc[leading_fj_mc_is_top_matched]
events_dict["W_matched"] = events_mc[leading_fj_mc_is_W_matched]
events_dict["unmatched"] = events_mc[leading_fj_mc_is_unmatched]

In [ ]:
events_dict["top_matched"]["leading_fj_tau21"]

## Plot tau21

In [ ]:
control_plot_vars = [
    ShapeVar(
        var="leading_fj_tau21", label=r"tau21", bins=list(np.arange(0, 1.05, 0.05)), reg=False
    ),
]

In [ ]:
ylims = {
    "2022": 5e4,
    "2022EE": 4e3,
    "2023-pre-BPix": 4e5,
}

In [ ]:
for year in ["2022EE"]:
    hists = {}
    for shape_var in control_plot_vars:
        print(shape_var)
        if shape_var.var not in hists:
            hists[shape_var.var] = utils.singleVarHist(
                events_dict,
                shape_var,
                weight_key="finalWeight",
            )

        bkgs = ["top_matched", "W_matched", "unmatched"]
        sigs = []

        plotting.ratioHistPlot(
            hists[shape_var.var],
            year,
            sigs,
            bkgs,
            name="test_wp_Wmass>60",
            show=True,
            log=False,
            bg_err=None,
            bg_order=["top_matched", "W_matched", "unmatched"],
            plot_data=True,
            plot_significance=False,
            significance_dir=shape_var.significance_dir,
            ylim=1.2e4,
            ylim_low=0,
        )

In [ ]:
plt.hist(
    events_dict["top_matched"]["leading_fj_tau21"],
    bins=list(np.arange(0, 1.05, 0.05)),
    weights=events_dict["top_matched"]["finalWeight"],
    label="top",
    histtype="step",
)
plt.hist(
    events_dict["W_matched"]["leading_fj_tau21"],
    bins=list(np.arange(0, 1.05, 0.05)),
    weights=events_dict["W_matched"]["finalWeight"],
    label="W",
    histtype="step",
)
plt.hist(
    events_dict["unmatched"]["leading_fj_tau21"],
    bins=list(np.arange(0, 1.05, 0.05)),
    weights=events_dict["unmatched"]["finalWeight"],
    label="unmatched",
    histtype="step",
)
plt.legend()